# TFG — Falsos positivos sobre fondo y matrices de confusión legibles

Cuaderno **autónomo**: no depende de `analisis_interpretabilidad_TFG.ipynb`.
Cubre tres peticiones del tutor:

1. **Imágenes de solo fondo** para completar la columna de *background* de las
   matrices de confusión (COCO *val2017* filtrado: sin `airplane`, `bird` ni `kite`).
2. **Letra grande** en títulos y números de todas las figuras.
3. Figuras y cifras listas para pegar en la memoria.

### Orden de ejecución
Ejecutar las celdas **en orden**. La §2 monta Drive y la §3 pide la clave de Roboflow
por `getpass` (no uses «Ejecutar todo» en la primera pasada: el `getpass` bloquea).

### Tiempo estimado
~25–40 min con GPU T4 (la descarga de COCO son ~780 MB; las validaciones son lo lento).

### Salidas en `OUT_DIR`
`background_fp.png` · `background_fp_ejemplos_{dronebird,unified}.png` ·
`confusion_{modelo}_320_bg.png` · `confusion_{modelo}_{320,640}[_counts].png` (repintadas)

## §1 · Instalación

In [ ]:
%pip install -q ultralytics roboflow

import ultralytics, numpy, matplotlib
print("ultralytics", ultralytics.__version__, "| numpy", numpy.__version__)

## §2 · Montaje de Drive y rutas

La salvaguarda `assert` evita el bug de la carpeta fantasma: si Drive no está bien
montado, la celda falla **antes** de crear ninguna carpeta.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

TFG = Path("/content/drive/MyDrive/TFG")          # OJO: Colab distingue mayusculas
assert (TFG / "models").exists(), f"No veo {TFG}/models. Revisa el montaje o la cuenta de Colab."

OUT_DIR = TFG / "analysis" / "figuras"            # AJUSTA si tu carpeta se llama distinto
assert OUT_DIR.exists(), f"No existe {OUT_DIR}. Crea la carpeta a mano en Drive y reejecuta."

PESOS = {
    "dronebird": TFG / "models/tfg_drone_bird/weights/best.pt",
    "unified":   TFG / "models/unified_aod4/run/weights/best.pt",
}
for k, v in PESOS.items():
    print(("[OK]    " if v.exists() else "[FALTA] ") + f"{k}: {v}")
print("[OK]    OUT_DIR:", OUT_DIR)

## §3 · Descarga de los datasets (Roboflow)

Necesarios para validar y obtener las matrices de confusión. `/content` es efímero,
así que hay que rebajarlos en cada sesión. La clave se pide por `getpass`: **no la
pegues en una celda**.

In [ ]:
import os
from getpass import getpass
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
rf = Roboflow(api_key=api_key)

# --- AOD-4 (modelo unificado): version 4 ---
ds_aod4 = (rf.workspace("airborne-object-detection")
             .project("airborne-object-detection-4-aod4")
             .version(4).download("yolov8"))

# --- Drone-vs-Birds (especialista): ultima version disponible ---
proj_db = rf.workspace("eliesaade-abmjh").project("drone-vs-birds-dvaqh")
try:
    versiones = [int(str(v.version).rstrip("/").split("/")[-1]) for v in proj_db.versions()]
    v_db = max(versiones)
except Exception as e:
    print("No pude autodetectar la version, uso la 1:", e)
    v_db = 1
print("Version dronebird:", v_db)
ds_db = proj_db.version(v_db).download("yolov8")

YAMLS = {
    "dronebird": Path(ds_db.location) / "data.yaml",
    "unified":   Path(ds_aod4.location) / "data.yaml",
}
for k, v in YAMLS.items():
    print(("[OK]    " if v.exists() else "[FALTA] ") + f"{k}: {v}")

## §4 · Parámetros del experimento y estilo de figuras

In [ ]:
import matplotlib.pyplot as plt
import yaml

# --- Letra grande en TODAS las figuras (peticion del tutor) ---
plt.rcParams.update({
    "savefig.dpi": 200, "figure.dpi": 140,
    "font.size": 16, "axes.titlesize": 22, "axes.titleweight": "bold",
    "axes.labelsize": 19, "xtick.labelsize": 16, "ytick.labelsize": 16,
    "legend.fontsize": 15, "figure.titlesize": 24,
})

CONF_APP = 0.25          # MISMO umbral que usa la app en produccion
TAMANOS  = (320,)        # anade 640 si quieres las dos resoluciones en el test de fondo
N_FONDO  = 1000          # nº de imagenes de fondo
EXT      = {".jpg", ".jpeg", ".png", ".bmp"}
TITULOS  = {"dronebird": "Especialista", "unified": "Unificado"}
BACKGROUND_DIR = Path("/content/background_coco")

def split_disponible(yaml_path, preferido="test"):
    d = yaml.safe_load(Path(yaml_path).read_text())
    for s in (preferido, "test", "val"):
        if d.get(s):
            return s
    raise RuntimeError(f"{yaml_path} no define test ni val")

SPLITS = {k: split_disponible(v) for k, v in YAMLS.items()}
print("Splits usados:", SPLITS)

## §5 · Conjunto de fondo: COCO *val2017* filtrado

Se eliminan todas las imágenes con anotaciones de `airplane`, `bird` o `kite`.
Se excluye `kite` a propósito: es un objeto pequeño volando contra cielo, así que
detectarlo no sería inequívocamente un error. Con ese filtro, **toda** detección
sobre el conjunto es por construcción un falso positivo y no hace falta revisión
manual.

In [ ]:
import json, random, shutil

COCO_DIR = Path("/content/coco")

if not (COCO_DIR / "annotations/instances_val2017.json").exists():
    print("Descargando COCO val2017 (~780 MB + anotaciones)...")
    os.system("wget -q -c http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip")
    os.system("wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/ann.zip")
    os.system("unzip -q -o /content/val2017.zip -d /content/coco")
    os.system("unzip -q -o /content/ann.zip     -d /content/coco")

ann = json.load(open(COCO_DIR / "annotations/instances_val2017.json"))
EXCLUIR = {5, 16, 38}                       # airplane, bird, kite
cat = {c["id"]: c["name"] for c in ann["categories"]}
print("Categorias excluidas:", [cat[i] for i in sorted(EXCLUIR)])

malas   = {a["image_id"] for a in ann["annotations"] if a["category_id"] in EXCLUIR}
limpias = [im for im in ann["images"] if im["id"] not in malas]
print(f"COCO val2017: {len(ann['images'])} imgs -> {len(limpias)} sin objetos aereos")

random.seed(0)
sel = random.sample(limpias, min(N_FONDO, len(limpias)))

if BACKGROUND_DIR.exists():
    shutil.rmtree(BACKGROUND_DIR)
BACKGROUND_DIR.mkdir(parents=True)
for im in sel:
    shutil.copy(COCO_DIR / "val2017" / im["file_name"], BACKGROUND_DIR / im["file_name"])

print(f"Conjunto de fondo listo: {len(list(BACKGROUND_DIR.glob('*')))} imagenes")

## §6 · Helper: matrices de confusión con tipografía legible

Ultralytics genera el PNG con tamaños de letra fijos, así que `rcParams` no le afecta.
Aquí se repinta a partir de la matriz numérica (`results.confusion_matrix.matrix`).

La celda inferior derecha se marca **`n/d`**: es «fondo predicho como fondo», es decir
los verdaderos negativos, que en detección no están definidos. Va a seguir vacía por
mucho fondo que añadas, y conviene saber explicarlo en la defensa.

In [ ]:
import numpy as np

def plot_cm(matrix, names, out_path, title, normalize=False, mask_corner=True):
    m = np.asarray(matrix, dtype=float)
    labels = list(names) + ["background"]
    n = len(labels)
    if normalize:
        s = m.sum(0, keepdims=True)
        m = np.divide(m, s, out=np.zeros_like(m), where=s > 1e-9)
    fig, ax = plt.subplots(figsize=(1.7 * n + 4, 1.5 * n + 3.5))
    im = ax.imshow(m, cmap="Blues", vmin=0, vmax=1 if normalize else max(m.max(), 1))
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=18)
    ax.set_yticklabels(labels, fontsize=18)
    ax.set_xlabel("Clase real", fontsize=20, fontweight="bold")
    ax.set_ylabel("Clase predicha", fontsize=20, fontweight="bold")
    ax.set_title(title, fontsize=24, fontweight="bold", pad=16)
    thr = (m.max() if m.max() > 0 else 1) * 0.6
    for i in range(n):
        for j in range(n):
            if mask_corner and i == n - 1 and j == n - 1:
                txt, col = "n/d", "0.35"
            else:
                txt = f"{m[i, j]:.2f}" if normalize else f"{int(round(m[i, j]))}"
                col = "white" if m[i, j] > thr else "black"
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=19, fontweight="bold", color=col)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=16)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("guardada:", out_path)

def nombres_de(model):
    return [model.names[i] for i in sorted(model.names)]

## §7 · Prueba de falsos positivos sobre fondo puro

In [ ]:
import cv2
from ultralytics import YOLO

def fp_test(key, imgsz=320, conf=CONF_APP, batch=32):
    model = YOLO(str(PESOS[key]))
    names = nombres_de(model)
    imgs  = sorted(p for p in BACKGROUND_DIR.glob("*") if p.suffix.lower() in EXT)
    por_clase = {n: 0 for n in names}
    con_deteccion, ejemplos = 0, []
    for i in range(0, len(imgs), batch):
        lote = imgs[i:i + batch]
        res = model.predict([str(p) for p in lote], imgsz=imgsz, conf=conf, verbose=False)
        for p, r in zip(lote, res):
            b = r.boxes
            if b is None or len(b) == 0:
                continue
            con_deteccion += 1
            for c, cf, xyxy in zip(b.cls.tolist(), b.conf.tolist(), b.xyxy.tolist()):
                por_clase[names[int(c)]] += 1
                ejemplos.append((float(cf), names[int(c)], str(p), xyxy))
    total = sum(por_clase.values())
    out = dict(key=key, imgsz=imgsz, conf=conf, n_img=len(imgs), names=names,
               por_clase=por_clase, con_deteccion=con_deteccion, total=total,
               tasa_img=100.0 * con_deteccion / max(len(imgs), 1),
               fp_por_img=total / max(len(imgs), 1),
               ejemplos=sorted(ejemplos, key=lambda x: -x[0]))
    print(f"[{key} @{imgsz}] {len(imgs)} imgs | {con_deteccion} con deteccion "
          f"({out['tasa_img']:.1f} %) | {total} FP ({out['fp_por_img']:.3f} FP/img) | {por_clase}")
    return out

RES_FP = {}
for sz in TAMANOS:
    for key in ("dronebird", "unified"):
        RES_FP[(key, sz)] = fp_test(key, imgsz=sz)

## §8 · Figura `background_fp.png`

In [ ]:
SZ = TAMANOS[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, key in zip(axes, ("dronebird", "unified")):
    r = RES_FP[(key, SZ)]
    clases  = list(r["por_clase"].keys())
    valores = [r["por_clase"][c] for c in clases]
    ax.bar(clases, valores, color="#2f6fb3", edgecolor="black", linewidth=0.8)
    for x, v in enumerate(valores):
        ax.text(x, v, str(v), ha="center", va="bottom", fontsize=18, fontweight="bold")
    ax.set_title(f"{TITULOS[key]} · {SZ} px", fontsize=22, fontweight="bold")
    ax.set_ylabel("Falsos positivos", fontsize=19)
    ax.set_ylim(0, max(max(valores), 1) * 1.25)
    ax.tick_params(axis="x", labelsize=17)
    ax.grid(axis="y", alpha=0.3)
    ax.text(0.02, 0.95,
            f"{r['con_deteccion']}/{r['n_img']} imagenes con deteccion "
            f"({r['tasa_img']:.1f} %)\n{r['fp_por_img']:.3f} FP/imagen · conf >= {r['conf']}",
            transform=ax.transAxes, va="top", fontsize=15,
            bbox=dict(boxstyle="round", fc="white", ec="0.6"))
fig.suptitle("Falsos positivos sobre fondo sin objetos aereos (COCO val2017 filtrado)",
             fontsize=24, fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "background_fp.png", dpi=200, bbox_inches="tight")
plt.show()
print("guardada:", OUT_DIR / "background_fp.png")

## §9 · Figuras `background_fp_ejemplos_*.png`

In [ ]:
def montaje_fp(key, sz=None, n=8, margen=0.15):
    sz = sz or SZ
    r = RES_FP[(key, sz)]
    top = r["ejemplos"][:n]
    if not top:
        print(f"{key}: sin falsos positivos, no se genera montaje")
        return
    filas = 2
    cols  = int(np.ceil(len(top) / filas))
    fig, axes = plt.subplots(filas, cols, figsize=(3.2 * cols, 3.6 * filas))
    ejes = np.ravel(np.atleast_2d(axes))
    for ax, (cf, cls, path, (x1, y1, x2, y2)) in zip(ejes, top):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mx, my = (x2 - x1) * margen, (y2 - y1) * margen
        x1, y1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
        x2, y2 = min(w, int(x2 + mx)), min(h, int(y2 + my))
        ax.imshow(img[y1:y2, x1:x2])
        ax.set_title(f"{cls} {cf:.2f}", fontsize=18, fontweight="bold")
        ax.axis("off")
    for ax in ejes[len(top):]:
        ax.axis("off")
    fig.suptitle(f"{TITULOS[key]}: falsos positivos con mayor confianza",
                 fontsize=22, fontweight="bold")
    fig.tight_layout()
    out = OUT_DIR / f"background_fp_ejemplos_{key}.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print("guardada:", out)

for key in ("dronebird", "unified"):
    montaje_fp(key)

## §10 · Matrices de confusión con fondo real

Se construye un split mezclado: las imágenes del conjunto de prueba **más** las de
fondo con un `.txt` de etiquetas **vacío** (así Ultralytics las trata como fondo puro).
Los avisos de «imágenes sin etiquetas» son esperados.

> **Importante:** `model.val(...)` debe llamarse con `plots=True`. Ultralytics solo
> alimenta `confusion_matrix` cuando esa opción está activa; con `plots=False` la
> matriz sale **entera a cero** sin dar ningún error.
>
> Igual de importante: hay que pasar `conf=0.25`. Por defecto `val()` usa
> `conf=0.001`, y las versiones recientes de Ultralytics ya no lo redondean a
> `0.25` para la matriz de confusión: la columna de *background* se llena de
> detecciones de confianza ínfima y sale inflada ~50 veces.

In [ ]:
def resolver_split(yaml_path, split):
    # Devuelve (dict, dir_imagenes, dir_etiquetas) del split pedido.
    # Los data.yaml de Roboflow escriben las rutas como '../test/images'.
    # Se usa primero el resolutor oficial de Ultralytics y, si falla, se
    # replica su regla: si la ruta con '../' no existe, se prueba sin el '../'.
    y = Path(yaml_path)
    d = yaml.safe_load(y.read_text())

    # --- via preferida: el propio resolutor de Ultralytics ---
    try:
        from ultralytics.data.utils import check_det_dataset
        info = check_det_dataset(str(y), autodownload=False)
        rel = info.get(split)
        if isinstance(rel, (list, tuple)):
            rel = rel[0]
        if rel:
            img_dir = Path(rel)
            if img_dir.exists():
                lbl_dir = (img_dir.parent / "labels" if img_dir.name == "images"
                           else Path(str(img_dir).replace("images", "labels")))
                return d, img_dir, lbl_dir
    except Exception as e:
        print("check_det_dataset no aplicable, uso resolucion manual:", e)

    # --- respaldo manual ---
    base = Path(d.get("path", y.parent))
    if not base.is_absolute():
        base = (y.parent / base).resolve()
    rel = d.get(split)
    if isinstance(rel, (list, tuple)):
        rel = rel[0]
    cand = [(base / rel).resolve()]
    if str(rel).startswith("../"):
        cand.append((base / str(rel)[3:]).resolve())
    cand.append((y.parent / str(rel).lstrip("./")).resolve())
    img_dir = next((c for c in cand if c.exists()), cand[0])
    lbl_dir = (img_dir.parent / "labels" if img_dir.name == "images"
               else Path(str(img_dir).replace("images", "labels")))
    return d, img_dir, lbl_dir

def enlazar(src, dst):
    try:
        os.symlink(src, dst)
    except (OSError, NotImplementedError):
        shutil.copy(src, dst)

def construir_mezclado(key):
    d, img_dir, lbl_dir = resolver_split(YAMLS[key], SPLITS[key])
    assert img_dir.exists(), (f"no encuentro imagenes: {img_dir}\n"
                              f"Contenido de {Path(YAMLS[key]).parent}: "
                              f"{[p.name for p in Path(YAMLS[key]).parent.iterdir()]}")
    dst = Path(f"/content/mixed_{key}")
    if dst.exists():
        shutil.rmtree(dst)
    (dst / "images").mkdir(parents=True)
    (dst / "labels").mkdir(parents=True)

    n_orig = 0
    for p in sorted(img_dir.iterdir()):
        if p.suffix.lower() not in EXT:
            continue
        enlazar(p, dst / "images" / p.name)
        lbl = lbl_dir / (p.stem + ".txt")
        (dst / "labels" / (p.stem + ".txt")).write_text(lbl.read_text() if lbl.exists() else "")
        n_orig += 1

    n_bg = 0
    for p in sorted(BACKGROUND_DIR.iterdir()):
        if p.suffix.lower() not in EXT:
            continue
        enlazar(p, dst / "images" / f"bg_{p.name}")
        (dst / "labels" / f"bg_{p.stem}.txt").write_text("")
        n_bg += 1

    nombres = d["names"]
    nuevo = {"path": str(dst), "train": "images", "val": "images",
             "names": nombres, "nc": d.get("nc", len(nombres))}
    (dst / "data.yaml").write_text(yaml.safe_dump(nuevo, sort_keys=False))
    print(f"{key}: {n_orig} imgs de '{SPLITS[key]}' + {n_bg} de fondo -> {dst}")
    return dst / "data.yaml"

for key in ("dronebird", "unified"):
    yaml_mix = construir_mezclado(key)
    model = YOLO(str(PESOS[key]))
    nombres = nombres_de(model)
    for sz in TAMANOS:
        r = model.val(data=str(yaml_mix), imgsz=sz, split="val",
                      conf=CONF_APP, plots=True, verbose=False)
        cm = r.confusion_matrix.matrix
        assert cm.sum() > 0, ("La matriz de confusion ha salido VACIA. Comprueba que "
                              "model.val se llama con plots=True.")
        print(f"   columna background (FP): {cm[:-1, -1].astype(int).tolist()} "
              f"| diagonal: {np.diag(cm)[:-1].astype(int).tolist()}")
        plot_cm(cm, nombres,
                OUT_DIR / f"confusion_{key}_{sz}_bg.png",
                f"{TITULOS[key]} · {sz} px — conteos con fondo real")

## §11 · Repintado de las matrices sin fondo (letra grande)

Sobrescribe `confusion_{modelo}_{320,640}.png` y `..._counts.png` con los **mismos
nombres de fichero**, de modo que la memoria no necesita ningún cambio: basta con
copiar las nuevas a `figuras/` en Overleaf.

In [ ]:
for key in ("dronebird", "unified"):
    model = YOLO(str(PESOS[key]))
    nombres = nombres_de(model)
    for sz in (320, 640):
        r = model.val(data=str(YAMLS[key]), imgsz=sz, split=SPLITS[key],
                      conf=CONF_APP, plots=True, verbose=False)
        cm = r.confusion_matrix.matrix
        assert cm.sum() > 0, ("La matriz de confusion ha salido VACIA. Comprueba que "
                              "model.val se llama con plots=True.")
        print(f"   columna background (FP): {cm[:-1, -1].astype(int).tolist()} "
              f"| diagonal: {np.diag(cm)[:-1].astype(int).tolist()}")
        plot_cm(cm, nombres, OUT_DIR / f"confusion_{key}_{sz}_counts.png",
                f"{TITULOS[key]} · {sz} px — conteos")
        plot_cm(cm, nombres, OUT_DIR / f"confusion_{key}_{sz}.png",
                f"{TITULOS[key]} · {sz} px — normalizada", normalize=True)

## §12 · Cifras para `tab:background_fp`

Copia las dos filas que imprime esta celda dentro de la tabla
`tab:background_fp` del Apéndice A (sustituyendo los `\dots`).

In [ ]:
print("% --- filas para tab:background_fp ---")
for key in ("dronebird", "unified"):
    r = RES_FP[(key, SZ)]
    detalle = ", ".join(f"{c}: {v}" for c, v in r["por_clase"].items())
    print(f"{TITULOS[key]} & \\num{{{r['n_img']}}} & \\num{{{r['con_deteccion']}}} & "
          f"\\num{{{r['tasa_img']:.1f}}} & \\num{{{r['total']}}} & "
          f"\\num{{{r['fp_por_img']:.3f}}} \\\\   % {detalle}")

print("\n% --- desglose completo (por si lo citas en el texto) ---")
for (key, sz), r in RES_FP.items():
    print(f"{key} @{sz}: {r['por_clase']}")

## §13 · Descarga de las figuras nuevas

In [ ]:
import zipfile
from google.colab import files

NUEVAS = ["background_fp.png",
          "background_fp_ejemplos_dronebird.png",
          "background_fp_ejemplos_unified.png",
          "confusion_dronebird_320_bg.png",
          "confusion_unified_320_bg.png"]
NUEVAS += [f"confusion_{k}_{s}{suf}.png"
           for k in ("dronebird", "unified") for s in (320, 640) for suf in ("", "_counts")]

zip_path = "/content/figuras_background.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for n in NUEVAS:
        p = OUT_DIR / n
        if p.exists():
            z.write(p, n)
        else:
            print("[FALTA]", n)

print("zip listo:", zip_path)
files.download(zip_path)